# Spark Fundamentals & Data Processing Pipeline

This notebook demonstrates dataset loading, schema modification, missing value imputation, filtering, and regional aggregations using PySpark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, min, max, count, when
from pyspark.sql.types import IntegerType

spark = SparkSession.builder \
    .appName("SparkBasics") \
    .getOrCreate()

## 1. Load Dataset

In [ ]:
df = spark.read.csv("../data/dataset.csv", header=True, inferSchema=True)
df.show()

## 2. Schema Modification

In [ ]:
df_mod = df.withColumn("age", col("age").cast(IntegerType())) \
           .withColumnRenamed("join_date", "hiring_date")

df_mod.printSchema()

## 3. Data Cleaning & Imputation

In [ ]:
df_clean = df_mod.dropDuplicates()

df_clean = df_clean.withColumn("region", when(col("region") == "", "Unknown").otherwise(col("region")))

df_clean = df_clean.fillna({
    "age": 30,
    "department": "Unassigned",
    "salary": 0.0,
    "hiring_date": "1900-01-01"
})

df_clean.show()

## 4. Filtering Conditions

In [ ]:
df_filtered = df_clean.filter(
    (col("age") >= 25) & 
    (col("age") <= 40) & 
    (col("department") != "Unassigned")
)

df_filtered.show()

## 5. Grouping & Aggregations

In [ ]:
agg_df = df_clean.groupBy("department").agg(
    avg("salary").alias("avg_salary"),
    count("id").alias("employee_count"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
)

agg_filtered = agg_df.filter(col("employee_count") > 1)
agg_filtered.show()

## 6. Pipeline Execution & Export

In [ ]:
final_pipeline_df = df \
    .dropDuplicates() \
    .withColumnRenamed("join_date", "hiring_date") \
    .withColumn("region", when(col("region") == "", "Unknown").otherwise(col("region"))) \
    .fillna({"age": 30, "department": "Unassigned", "salary": 0.0, "hiring_date": "1900-01-01"}) \
    .filter(col("salary") > 0) \
    .groupBy("region") \
    .agg(sum("salary").alias("total_regional_salary")) \
    .orderBy(col("total_regional_salary").desc())

final_pipeline_df.show()

final_pipeline_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("../output/results.csv")

spark.stop()